In [1]:
import csv
import gc
import sys
import time
import json
import os
import random
import re
from contextlib import contextmanager
from copy import deepcopy
from typing import Callable, Optional, Any

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
import pandas as pd
import re
from pathlib import Path


INPUT_DIR = Path("/kaggle/input/datasets/yumiasakura/ml-cw-results-qwen")

# Папка для обработанных CSV
OUTPUT_DIR = Path("/kaggle/working/prepared_csvs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Поля, в которых условие может уже храниться отдельно
QUESTION_COLUMNS = [
    "question_text",
    "question",
    "problem",
    "query"
]

def clean_question_text(text):
    """Удаляет лишние пробелы и переносы строк."""
    if not text:
        return None

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    return text if text else None

def extract_from_prompt(prompt):
    """
    Извлекает условие текущей задачи.

    Если в prompt несколько Question: (few-shot),
    берёт последнее вхождение.

    Если Question: отсутствует, удаляет известные
    инструкции в конце обычного prompt.
    """

    if pd.isna(prompt) or not str(prompt).strip():
        return None, None

    prompt = str(prompt).strip()

    # Все маркеры Question:
    markers = list(
        re.finditer(
            r"\b(?:Question|Problem)\s*:\s*",
            prompt,
            flags=re.IGNORECASE
        )
    )

    if markers:
        # Последний Question: - условие текущей задачи,
        # а предыдущие обычно являются few-shot примерами.
        last_marker = markers[-1]

        question = prompt[last_marker.end():]

        # Убираем возможные инструкции после условия.
        stop_patterns = [
            r"\n\s*Solve step by step\b",
            r"\n\s*Reply with only\b",
            r"\n\s*At the end write\b",
            r"\n\s*Answer\s*:",
        ]

        for pattern in stop_patterns:
            match = re.search(
                pattern,
                question,
                flags=re.IGNORECASE
            )
            if match:
                question = question[:match.start()]

        question = clean_question_text(question)

        if question:
            return question, "prompt:last_question_marker"

        return None, "prompt:empty_question_marker"

    # --------------------------------------------------------
    # 2. Если Question: нет, удаляем инструкции в конце.
    # --------------------------------------------------------

    # Не пытаемся извлекать задачу из демонстрационного
    # блока, если в нём не удалось найти Question:.
    if re.search(r"\bExamples\s*:", prompt, flags=re.IGNORECASE):
        return None, "prompt:ambiguous_examples"

    instruction_patterns = [
        r"\s*Reply with only the final number.*$",
        r"\s*Reply with only the letter.*$",
        r"\s*Provide only the answer\.?$",
        r"\s*Solve step by step\..*$",
        r"\s*At the end write.*$",
    ]

    question = prompt

    for pattern in instruction_patterns:
        question = re.sub(
            pattern,
            "",
            question,
            flags=re.IGNORECASE | re.DOTALL
        )

    question = clean_question_text(question)

    if question:
        return question, "prompt:instruction_removed"

    return None, None

# ============================================================
# ОБРАБОТКА ОДНОЙ СТРОКИ
# ============================================================

def extract_question_text(row):
    """
    Сначала использует отдельные колонки с условием.
    Если их нет — извлекает условие из prompt.
    """

    for column in QUESTION_COLUMNS:
        value = row.get(column)

        if pd.notna(value) and str(value).strip():
            return (
                clean_question_text(value),
                f"column:{column}"
            )

    return extract_from_prompt(row.get("prompt"))

# ============================================================
# ОБРАБОТКА ОДНОГО CSV
# ============================================================

def prepare_csv(input_path, output_path):
    df = pd.read_csv(input_path, encoding="utf-8-sig")

    if "question_text" not in df.columns:
        df["question_text"] = None

    if "question_text_source" not in df.columns:
        df["question_text_source"] = None

    # Заполняем оба поля заново для каждой строки,
    # чтобы источник соответствовал фактическому извлечению.
    extracted = df.apply(extract_question_text, axis=1)

    df["question_text"] = [x[0] for x in extracted]
    df["question_text_source"] = [x[1] for x in extracted]

    # --------------------------------------------------------
    # Проверки
    # --------------------------------------------------------

    total = len(df)
    missing = df["question_text"].isna().sum()
    success = total - missing

    print("\n" + "=" * 60)
    print("Файл:", input_path.name)
    print("Строк:", total)
    print("Успешно извлечено:", success)
    print("Не удалось извлечь:", missing)

    print("\nИсточники:")
    print(df["question_text_source"].value_counts(dropna=False))

    # Показываем проблемные строки
    if missing:
        print("\nПервые строки без question_text:")
        columns_to_show = [
            c for c in ["question_id", "dataset", "mode", "prompt"]
            if c in df.columns
        ]

        print(
            df.loc[
                df["question_text"].isna(),
                columns_to_show
            ].head(5).to_string(index=False)
        )

    # Проверяем согласованность условия между режимами
    if "question_id" in df.columns:
        consistency = (
            df.dropna(subset=["question_text"])
              .groupby("question_id")["question_text"]
              .nunique()
        )

        inconsistent = consistency[consistency > 1]

        print(
            "\nВопросов с различающимся question_text "
            "между строками:",
            len(inconsistent)
        )

        if len(inconsistent):
            print("Примеры question_id:", inconsistent.head(10).index.tolist())

    # Сохраняем результат
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Сохранено:", output_path)

    return {
        "file": input_path.name,
        "rows": total,
        "extracted": success,
        "missing": missing,
        "output": str(output_path)
    }

# ============================================================
# ОБРАБОТКА ВСЕХ CSV В ПАПКЕ
# ============================================================

csv_files = sorted(INPUT_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"В папке {INPUT_DIR} не найдены CSV-файлы"
    )

reports = []

for src_path in csv_files:

    # Не обрабатываем повторно файлы, созданные скриптом.
    if src_path.stem.endswith("_prepared"):
        continue

    dst_path = OUTPUT_DIR / (
        src_path.stem + "_prepared.csv"
    )

    report = prepare_csv(src_path, dst_path)
    reports.append(report)

# ============================================================
# ОБЩИЙ ОТЧЁТ
# ============================================================

report_df = pd.DataFrame(reports)

print("\n" + "=" * 60)
print("ИТОГОВЫЙ ОТЧЁТ")

if not report_df.empty:
    print(report_df.to_string(index=False))

    print("\nВсего обработано строк:", report_df["rows"].sum())
    print("Всего извлечено:", report_df["extracted"].sum())
    print("Всего пропущено:", report_df["missing"].sum())

    report_df.to_csv(
        OUTPUT_DIR / "preprocessing_report.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\nВсе обработанные файлы находятся в:", OUTPUT_DIR)


Файл: results_test100_Qwen2.5-0.5B-Instruct.csv
Строк: 600
Успешно извлечено: 600
Не удалось извлечь: 0

Источники:
question_text_source
prompt:last_question_marker    400
prompt:instruction_removed     200
Name: count, dtype: int64

Вопросов с различающимся question_text между строками: 100
Примеры question_id: ['arc_1', 'arc_1000', 'arc_1010', 'arc_1025', 'arc_1039', 'arc_1043', 'arc_1052', 'arc_1059', 'arc_1080', 'arc_1083']
Сохранено: /kaggle/working/prepared_csvs/results_test100_Qwen2.5-0.5B-Instruct_prepared.csv

Файл: results_test100_Qwen2.5-3B-Instruct.csv
Строк: 600
Успешно извлечено: 600
Не удалось извлечь: 0

Источники:
question_text_source
prompt:last_question_marker    400
prompt:instruction_removed     200
Name: count, dtype: int64

Вопросов с различающимся question_text между строками: 100
Примеры question_id: ['arc_1', 'arc_1000', 'arc_1010', 'arc_1025', 'arc_1039', 'arc_1043', 'arc_1052', 'arc_1059', 'arc_1080', 'arc_1083']
Сохранено: /kaggle/working/prepared_csvs/res

In [3]:
# Числовые значения и извлечение ответов
import re
import json
import random
from copy import deepcopy

NUMBER_PATTERN = re.compile(
    r"(?<![\w.,])-?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?!\w)"
    r"|(?<![\w.,])-?\d+(?:\.\d+)?(?!\w)"
)
STEP_LABEL = re.compile(r"^\s*(?:step\s*\d+\s*[:.)-]?\s*|\d+[.)]\s+)", re.I)
ANSWER_CUE = "Answer:"
ANSWER_MAX_TOKENS = 24


def find_numbers(step):
    """Числа шага без номера самого шага."""
    label = STEP_LABEL.match(str(step))
    start = label.end() if label else 0
    return [m for m in NUMBER_PATTERN.finditer(str(step)) if m.start() >= start]


def extract_number_ex(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", str(text))
    if m:
        s, ok = m.group(1), True
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", str(text))
        s, ok = (nums[-1] if nums else None), False
    return (s.replace(",", "").rstrip("."), ok) if s else (None, False)


def extract_letter_ex(text):
    text = str(text)
    m = re.search(r"Answer:\s*\(?([A-E])\b", text, flags=re.I)
    if m:
        return m.group(1).upper(), True
    cands = re.findall(r"(?i:answer\s+is)\s*:?\s*\(?([A-E])\b", text)
    if not cands:
        cands = re.findall(r"\(([A-E])\)", text)
    return (cands[-1].upper(), False) if cands else (None, False)


def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False


def normalize_dataset(dataset):
    value = str(dataset).strip().lower()
    if value in {"gsm8k", "gsm-8k"}:
        return "gsm8k"
    if value.startswith("arc"):
        return "arc"
    return value


def extract_answer_ex(text, dataset):
    return extract_number_ex(text) if normalize_dataset(dataset) == "gsm8k" else extract_letter_ex(text)


def answers_equal(prediction, reference, dataset):
    if prediction is None or reference is None:
        return False
    if normalize_dataset(dataset) == "gsm8k":
        return same_number(prediction, reference)
    return str(prediction).strip().upper() == str(reference).strip().upper()


def split_steps(text):
    lines = [s.strip() for s in str(text).split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]

In [4]:
def parse_steps(value):
    """Читает steps как JSON-список; при старом текстовом формате делит по абзацам."""
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if value is None or not str(value).strip():
        return []
    value = str(value).strip()
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except json.JSONDecodeError:
        pass
    return split_steps(value)


def serialize_steps(steps):
    return json.dumps(list(steps), ensure_ascii=False)


def extract_question_text(row):
    """Берёт только отдельное поле условия, никогда не сканирует весь few-shot prompt."""
    for key in ("question_text", "question", "problem", "query"):
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value).strip(), key
    return None, None


def question_numbers_from_text(question_text):
    """Числа только из явно выделенного текста текущей задачи."""
    if question_text is None:
        return None
    values = set()
    for match in NUMBER_PATTERN.finditer(str(question_text)):
        try:
            values.add(float(match.group().replace(",", "")))
        except ValueError:
            continue
    return values


def _format_like(old_text, value):
    decimals = len(old_text.split(".")[1]) if "." in old_text else 0
    return f"{value:,.{decimals}f}" if "," in old_text else f"{value:.{decimals}f}"


def corrupt_value(old_value, decimals, rng):
    unit = 10 ** (-decimals)
    if old_value == 0:
        return rng.choice([1, 2, 3, 5]) * unit
    for _ in range(40):
        fraction = rng.choice([0.1, 0.25, 0.5])
        delta = rng.choice([-1, 1]) * max(unit, abs(old_value) * fraction)
        candidate = round(old_value + delta, decimals)
        if candidate != old_value and candidate * old_value > 0:
            return candidate
    return None


def replace_number(steps, step_index, number_index, rng):
    if not 0 <= step_index < len(steps):
        return None
    matches = find_numbers(steps[step_index])
    if not 0 <= number_index < len(matches):
        return None
    match = matches[number_index]
    old_text = match.group()
    try:
        old_value = float(old_text.replace(",", ""))
    except ValueError:
        return None
    decimals = len(old_text.split(".")[1]) if "." in old_text else 0
    new_value = corrupt_value(old_value, decimals, rng)
    if new_value is None:
        return None
    new_text = _format_like(old_text, new_value)
    if new_text == old_text:
        return None
    modified = list(steps)
    modified[step_index] = steps[step_index][:match.start()] + new_text + steps[step_index][match.end():]
    shift = (new_value - old_value) / old_value if old_value != 0 else None
    return modified, old_text, new_text, shift


def truncate_steps(steps, keep_count):
    """Оставляет ровно первые keep_count шагов; допустимо 0 <= keep_count < n."""
    if not 0 <= keep_count < len(steps):
        raise ValueError("keep_count must be in [0, len(steps)-1]")
    return list(steps[:keep_count])


def swap_adjacent(steps, index):
    if not 0 <= index < len(steps) - 1:
        return None
    changed = list(steps)
    changed[index], changed[index + 1] = changed[index + 1], changed[index]
    if changed == list(steps):
        return None
    return changed


def distinct_permutations(steps, rng, n_permutations=3, max_tries=500):
    """Возвращает уникальные неидентичные перестановки (с сохранением всех шагов)."""
    original = list(steps)
    seen = {tuple(original)}
    result = []
    for _ in range(max_tries):
        order = list(range(len(original)))
        rng.shuffle(order)
        candidate = [original[i] for i in order]
        key = tuple(candidate)
        if key not in seen and candidate != original:
            seen.add(key)
            result.append((order, candidate))
            if len(result) >= n_permutations:
                break
    return result


def relative_position(index, n_steps):
    return index / (n_steps - 1) if n_steps > 1 else 0.0


def build_variants(steps, rng, given_numbers=None, max_numbers=None, n_permutations=3):
    """Все варианты подаются в одном режиме: продолжить после полной цепочки и Answer:."""
    steps = list(steps)
    n = len(steps)
    if n < 2:
        return []
    variants = []

    def add(kind, changed, **meta):
        variants.append({"perturbation": kind, "modified_steps": list(changed), **meta})

    # Один контроль на вопрос: полная исходная цепочка без вмешательства.
    add("control", steps, target_step_index=None, relative_position=None,
        kept_fraction=1.0, removed_step_count=0, permutation_order=None)

    targets = [(si, ni) for si, step in enumerate(steps) for ni in range(len(find_numbers(step)))]
    if max_numbers is not None and len(targets) > max_numbers:
        targets = sorted(rng.sample(targets, max_numbers))
    given = set(given_numbers) if given_numbers is not None else None
    for si, ni in targets:
        result = replace_number(steps, si, ni, rng)
        if result is None:
            continue
        modified, old, new, shift = result
        old_value = float(old.replace(",", ""))
        nums_in_step = find_numbers(steps[si])
        add("number_substitution", modified,
            target_step_index=si, relative_position=relative_position(si, n),
            kept_fraction=1.0, removed_step_count=0,
            target_number_index=ni, old_number=old, new_number=new,
            relative_shift=shift,
            number_is_given=(old_value in given) if given is not None else None,
            is_last_number_in_step=(ni == len(nums_in_step) - 1),
            permutation_order=None)

    # Префиксная обрезка: сохраняются первые k шагов, k=0,...,n-1.
    for keep in range(n):
        add("truncation", truncate_steps(steps, keep),
            target_step_index=None, relative_position=None,
            kept_fraction=keep / n, removed_step_count=n - keep,
            truncation_boundary=keep, permutation_order=None)

    # Перестановки соседей; target_step_index — левая позиция обмена.
    for i in range(n - 1):
        changed = swap_adjacent(steps, i)
        if changed is not None:
            add("adjacent_swap", changed, target_step_index=i,
                relative_position=relative_position(i, n), kept_fraction=1.0,
                removed_step_count=0, permutation_order=None)

    # Случайные полные перестановки с сохранением всех исходных шагов.
    for order, changed in distinct_permutations(steps, rng, n_permutations):
        add("step_permutation", changed, target_step_index=None,
            relative_position=None, kept_fraction=1.0, removed_step_count=0,
            permutation_order=json.dumps(order))

    return variants


def build_assistant_prefix(modified_steps):
    chain = "\n\n".join(modified_steps)
    return (chain + "\n\n" + ANSWER_CUE) if chain else ANSWER_CUE


In [5]:
FIELDNAMES = [
    "question_id", "dataset", "model", "mode",
    "perturbation_id", "perturbation", "n_steps",
    "target_step_index", "relative_position", "kept_fraction", "removed_step_count",
    "truncation_boundary", "target_number_index", "old_number", "new_number", "relative_shift",
    "number_is_given", "is_last_number_in_step", "permutation_order",
    "question_text_source", "original_steps", "perturbed_steps", "assistant_prefix",
    "original_answer", "gold_answer", "regenerated_text", "regenerated_answer",
    "parse_success", "format_ok", "answer_change_status", "answer_changed",
    "regenerated_correct", "original_correct", "control_answer_changed", "control_adjusted_change",
]


def _row_key(row):
    return (str(row.get("question_id", "")), str(row.get("dataset", "")),
            str(row.get("model", "")), str(row.get("mode", "")))


def _truthy(value):
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def _answer_change(prediction, original, dataset):
    if original is None or not str(original).strip():
        return None, "missing_original"
    if prediction is None or not str(prediction).strip():
        return None, "unparseable_regenerated"
    changed = not answers_equal(prediction, original, dataset)
    return bool(changed), "changed" if changed else "unchanged"


def run_experiment(input_csv, output_csv, generate_fn: Callable,
                   seed=42, max_rows=None, save_every=10, append_existing=False,
                   skip_truncated=True, only_correct=True, max_numbers=None,
                   n_permutations=3, dry_run=False):
    """Единый answer-only эксперимент; контроль и все вмешательства парны по вопросу.

    dry_run=True: только план (файл не трогается, generate_fn не вызывается);
    возвращает число вопросов и генераций, которые осталось выполнить."""
    rng = random.Random(seed)
    tasks = []
    with open(input_csv, "r", encoding="utf-8-sig", newline="") as f:
        for row_number, row in enumerate(csv.DictReader(f)):
            if max_rows is not None and row_number >= max_rows:
                break
            original_steps = parse_steps(row.get("steps", ""))
            if len(original_steps) < 2:
                continue
            if skip_truncated and _truthy(row.get("truncated", "")):
                continue
            if only_correct and not _truthy(row.get("correct", "")):
                continue
            question_text, question_text_source = extract_question_text(row)
            given_numbers = question_numbers_from_text(question_text)
            variants = build_variants(original_steps, rng, given_numbers,
                                      max_numbers=max_numbers, n_permutations=n_permutations)
            tasks.append((row, original_steps, variants, question_text_source))

    append = append_existing and os.path.exists(output_csv) and os.path.getsize(output_csv) > 0
    done = set()
    if append:
        with open(output_csv, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames != FIELDNAMES:
                raise ValueError("Схема CSV изменилась; используйте новый output_csv или append_existing=False.")
            done = {_row_key(r) for r in reader}
    todo = [t for t in tasks if _row_key(t[0]) not in done]
    total = sum(len(v) for _, _, v, _ in todo)
    print(f"Вопросов: {len(tasks)}, уже сохранено: {len(tasks)-len(todo)}, к обработке: {len(todo)}, генераций: {total}")

    if dry_run:
        return {"questions": len(tasks), "pending_questions": len(todo), "pending_generations": total}

    out = open(output_csv, "a" if append else "w",
               encoding="utf-8" if append else "utf-8-sig", newline="")
    writer = csv.DictWriter(out, fieldnames=FIELDNAMES)
    if not append:
        writer.writeheader(); out.flush()
    pending, results = [], []
    saved = 0
    bar = tqdm(total=total, desc="Генерация", unit="ген")

    def flush():
        nonlocal saved
        if pending:
            writer.writerows(pending); out.flush(); os.fsync(out.fileno())
            saved += len(pending); pending.clear()

    try:
        for task_idx, (row, original_steps, variants, question_text_source) in enumerate(todo, 1):
            dataset = normalize_dataset(row.get("dataset", ""))
            original_answer = row.get("extracted_answer")
            gold_answer = row.get("gold_answer")
            question_results = []
            for variant_id, variant in enumerate(variants):
                prefix = build_assistant_prefix(variant["modified_steps"])
                generated_text = generate_fn(row.get("prompt", ""), row, prefix, ANSWER_MAX_TOKENS)
                prediction, format_ok = extract_answer_ex(ANSWER_CUE + str(generated_text), dataset)
                parse_success = prediction is not None and str(prediction).strip() != ""
                answer_changed, change_status = _answer_change(prediction, original_answer, dataset)
                regenerated_correct = (answers_equal(prediction, gold_answer, dataset)
                                        if prediction is not None and gold_answer is not None else None)
                record = {k: row.get(k) for k in ("question_id", "dataset", "model", "mode")}
                record.update({
                    "perturbation_id": variant_id, "perturbation": variant["perturbation"],
                    "n_steps": len(original_steps),
                    "target_step_index": variant.get("target_step_index"),
                    "relative_position": variant.get("relative_position"),
                    "kept_fraction": variant.get("kept_fraction"),
                    "removed_step_count": variant.get("removed_step_count"),
                    "truncation_boundary": variant.get("truncation_boundary"),
                    "target_number_index": variant.get("target_number_index"),
                    "old_number": variant.get("old_number"), "new_number": variant.get("new_number"),
                    "relative_shift": variant.get("relative_shift"),
                    "number_is_given": variant.get("number_is_given"),
                    "is_last_number_in_step": variant.get("is_last_number_in_step"),
                    "permutation_order": variant.get("permutation_order"),
                    "question_text_source": question_text_source,
                    "original_steps": serialize_steps(original_steps),
                    "perturbed_steps": serialize_steps(variant["modified_steps"]),
                    "assistant_prefix": prefix, "original_answer": original_answer,
                    "gold_answer": gold_answer, "regenerated_text": generated_text,
                    "regenerated_answer": prediction, "parse_success": parse_success,
                    "format_ok": bool(format_ok), "answer_change_status": change_status,
                    "answer_changed": answer_changed, "regenerated_correct": regenerated_correct,
                    "original_correct": _truthy(row.get("correct", "")),
                    "control_answer_changed": None, "control_adjusted_change": None,
                })
                question_results.append(record); bar.update(1)
            # Контроль один на вопрос. Парное сравнение каждого вмешательства с ним.
            control = next((x for x in question_results if x["perturbation"] == "control"), None)
            control_value = control["answer_changed"] if control else None
            for record in question_results:
                if record["perturbation"] != "control":
                    record["control_answer_changed"] = control_value
                    if record["answer_changed"] is not None and control_value is not None:
                        record["control_adjusted_change"] = int(record["answer_changed"]) - int(control_value)
            pending.extend(question_results); results.extend(question_results)
            if task_idx % max(1, save_every) == 0:
                flush()
    finally:
        flush(); bar.close(); out.close()
    print(f"Готово. Сохранено вариантов: {len(results)}; файл: {output_csv}")
    return results


## Несколько моделей

Для каждой модели из `MODEL_CONFIGS` нужен **свой входной CSV с цепочками, сгенерированными этой же моделью** (результаты части A). Подмена и сравнение ответов имеют смысл только для собственных цепочек модели.
Результаты каждой модели пишутся в отдельный файл `perturbation_results_<tag>.csv`; модели выполняются по очереди, после каждой веса выгружаются из CUDA.

In [6]:
model = None
tokenizer = None
CURRENT_MODEL_NAME = None

# Приблизительный размер весов в fp16, ГБ (для проверки, поместится ли модель до загрузки)
APPROX_FP16_GB = {
    "Qwen/Qwen2.5-0.5B-Instruct": 1.0,
    "Qwen/Qwen2.5-3B-Instruct": 6.2,
    "Qwen/Qwen2.5-7B-Instruct": 15.2,
}


def gpu_memory_report(prefix=""):
    if not torch.cuda.is_available():
        print(prefix + "CUDA недоступна")
        return
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        allocated = torch.cuda.memory_allocated(i)
        print(f"{prefix}cuda:{i} {torch.cuda.get_device_name(i)}: "
              f"занято тензорами {allocated / 2**30:.2f} ГБ | свободно {free / 2**30:.2f} из {total / 2**30:.2f} ГБ")


def free_gpu_memory():
    # Очистка кэша 
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gc.collect()


def unload_model():
    """Выгружает модель и токенайзер и освобождает память GPU."""
    global model, tokenizer, CURRENT_MODEL_NAME
    name = CURRENT_MODEL_NAME
    model = None
    tokenizer = None
    CURRENT_MODEL_NAME = None
    sys.last_traceback = None   # трейсбэк прошлой ошибки удерживает ссылки на тензоры
    free_gpu_memory()
    if name:
        print(f"Модель выгружена: {name}")


def load_model(model_name, quantization=None, dtype=torch.float16, force=False):
    """
    Загружает модель в глобальные model / tokenizer (предыдущая выгружается).
    quantization: None (fp16) | "8bit" | "4bit" (нужен bitsandbytes; числа нельзя напрямую
    сравнивать с fp16-прогонами других моделей).
    """
    global model, tokenizer, CURRENT_MODEL_NAME
    if CURRENT_MODEL_NAME is not None or model is not None:
        unload_model()

    need = APPROX_FP16_GB.get(model_name)
    if need and quantization is None and torch.cuda.is_available() and not force:
        free_total = sum(torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())) / 2**30
        if free_total < need * 1.1:
            raise RuntimeError(
                f"{model_name}: веса fp16 ~{need} ГБ, свободно {free_total:.1f} ГБ по всем GPU. "
                "Возьмите 2 GPU (device_map='auto' разделит модель), задайте quantization='8bit'/'4bit' "
                "или force=True."
            )

    kwargs = {"device_map": "auto"}
    if quantization == "8bit":
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif quantization == "4bit":
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=dtype, bnb_4bit_quant_type="nf4")
    else:
        kwargs["dtype"] = dtype

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
        model.eval()
        CURRENT_MODEL_NAME = model_name
    except BaseException:
        unload_model()       # не оставляем частично загруженные веса в памяти
        raise
    print("Устройство:", model.device, "| модель:", model_name,
          "| квантование:", quantization or "нет (fp16)")


@contextmanager
def loaded_model(model_name, **kwargs):
    """with loaded_model(name): ... - гарантированная выгрузка, в том числе при ошибке и Ctrl+C."""
    try:
        load_model(model_name, **kwargs)
        yield
    finally:
        unload_model()

In [7]:
def generate(model, tokenizer, user_prompt, max_new_tokens, assistant_prefix=""):
    """assistant_prefix - начало ответа ассистента, которое модель продолжает."""
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text += assistant_prefix
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [8]:
def qwen_generate(prompt, row, assistant_prefix="", max_new_tokens=700):
    if model is None or tokenizer is None:
        raise RuntimeError("Модель не загружена: используйте load_model() или loaded_model().")
    generated_text, n_tokens = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
        assistant_prefix=assistant_prefix,
    )
    return generated_text

In [9]:
RESULTS_DIR = "/kaggle/working"
# Входной CSV каждой модели (результаты части A для этой же модели).
INPUT_TEMPLATE = ("/kaggle/working/prepared_csvs/"
                  "results_test100_{short}_prepared.csv")   # short уже содержит "-Instruct"

MODEL_CONFIGS = [
    {"name": "Qwen/Qwen2.5-0.5B-Instruct", "tag": "0.5B"},
    {"name": "Qwen/Qwen2.5-3B-Instruct", "tag": "3B"},
    # fp16 ~15 ГБ: на одной 16-ГБ карте не поместится; нужны 2 GPU (Kaggle 2xT4) или "quantization": "8bit"/"4bit".
    {"name": "Qwen/Qwen2.5-7B-Instruct", "tag": "7B"},
    # Явный путь к другому входному файлу: {"name": ..., "tag": ..., "input_csv": "/path/file.csv"}
]


def input_path(cfg):
    return cfg.get("input_csv") or INPUT_TEMPLATE.format(short=cfg["name"].split("/")[-1])


def output_path(cfg):
    return os.path.join(RESULTS_DIR, f"perturbation_results_{cfg['tag']}.csv")


RESULT_FILES = {cfg["tag"]: output_path(cfg) for cfg in MODEL_CONFIGS}
os.makedirs(RESULTS_DIR, exist_ok=True)


def run_all_models(configs=None, only=None, append_existing=True, seed=42, max_rows=None,
                   save_every=1, skip_truncated=True, only_correct=True,
                   max_numbers=None, n_permutations=3):
    """
    Модели обрабатываются по очереди: загрузка -> run_experiment -> выгрузка из CUDA.
    only: список тегов/имён для выборочного запуска, например ["3B"].
    append_existing=True: незавершённая модель продолжается с места остановки, готовая пропускается
    (False - файл модели создаётся заново).
    Ошибка нехватки памяти или отсутствие входного файла не останавливают остальные модели.
    """
    configs = configs or MODEL_CONFIGS
    status_rows = []
    for cfg in configs:
        name, tag = cfg["name"], cfg["tag"]
        if only and tag not in only and name not in only:
            continue
        in_csv, out_csv = input_path(cfg), output_path(cfg)
        print(f"\n===== {name} ({tag}) =====")
        if not os.path.exists(in_csv):
            print(f"Нет входного файла: {in_csv}")
            folder = os.path.dirname(in_csv)
            similar = [f for f in sorted(os.listdir(folder)) if name.split("/")[-1] in f] if os.path.isdir(folder) else []
            print("Похожие файлы в папке:", similar[:6] if similar else "нет (папка пуста или не существует)")
            status_rows.append({"model": name, "tag": tag, "status": "нет входного файла",
                                "output_csv": out_csv})
            continue

        common = dict(input_csv=in_csv, output_csv=out_csv, seed=seed, max_rows=max_rows,
                      save_every=save_every, append_existing=append_existing,
                      skip_truncated=skip_truncated, only_correct=only_correct,
                      max_numbers=max_numbers, n_permutations=n_permutations)
        plan = run_experiment(generate_fn=None, dry_run=True, **common)
        if plan["pending_questions"] == 0:
            print("Все вопросы уже обработаны, пропускаем.")
            status_rows.append({"model": name, "tag": tag, "status": "готово ранее",
                                "output_csv": out_csv})
            continue

        started = time.time()
        status = "ok"
        try:
            with loaded_model(name, quantization=cfg.get("quantization"), force=cfg.get("force", False)):
                gpu_memory_report("после загрузки: ")
                run_experiment(generate_fn=qwen_generate, **common)
        except RuntimeError as exc:          # включает torch.cuda.OutOfMemoryError
            status = f"ошибка: {type(exc).__name__}: {str(exc)[:200]}"
            print(status)
        gpu_memory_report("после выгрузки: ")
        status_rows.append({"model": name, "tag": tag, "status": status,
                            "pending_generations": plan["pending_generations"],
                            "minutes": round((time.time() - started) / 60, 1),
                            "output_csv": out_csv})
    return pd.DataFrame(status_rows)

In [10]:
# Запуск всех моделей. Для выборочного запуска: run_all_models(only=["3B"]).
# Каждая модель пишет в свой файл perturbation_results_<tag>.csv; после модели веса выгружаются из CUDA.
run_status = run_all_models()
display(run_status)


===== Qwen/Qwen2.5-0.5B-Instruct (0.5B) =====
Вопросов: 148, уже сохранено: 0, к обработке: 148, генераций: 3802


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (30 outputs left out)

Устройство: cuda:0 | модель: Qwen/Qwen2.5-7B-Instruct | квантование: нет (fp16)
после загрузки: cuda:0 Tesla T4: занято тензорами 7.10 ГБ | свободно 7.31 из 14.56 ГБ
после загрузки: cuda:1 Tesla T4: занято тензорами 7.10 ГБ | свободно 7.31 из 14.56 ГБ
Вопросов: 300, уже сохранено: 0, к обработке: 300, генераций: 7277


Генерация:   0%|          | 0/7277 [00:00<?, ?ген/s]

Готово. Сохранено вариантов: 7277; файл: /kaggle/working/perturbation_results_7B.csv
Модель выгружена: Qwen/Qwen2.5-7B-Instruct
после выгрузки: cuda:0 Tesla T4: занято тензорами 0.01 ГБ | свободно 14.41 из 14.56 ГБ
после выгрузки: cuda:1 Tesla T4: занято тензорами 0.01 ГБ | свободно 14.41 из 14.56 ГБ


,model,tag,status,pending_generations,minutes,output_csv
0,Qwen/Qwen2.5-0.5B-Instruct,0.5B,ok,3802,21.3,/kaggle/working/perturbation_results_0.5B.csv
1,Qwen/Qwen2.5-3B-Instruct,3B,ok,7900,74.6,/kaggle/working/perturbation_results_3B.csv
2,Qwen/Qwen2.5-7B-Instruct,7B,ok,7277,110.9,/kaggle/working/perturbation_results_7B.csv


In [11]:
import numpy as np
import pandas as pd


def _to_bool_or_nan(value):
    if pd.isna(value): return np.nan
    s = str(value).strip().lower()
    if s in {"true", "1", "yes"}: return 1.0
    if s in {"false", "0", "no"}: return 0.0
    return np.nan


def load_results(path):
    frame = pd.read_csv(path, encoding="utf-8-sig")
    for col in ("answer_changed", "regenerated_correct", "format_ok", "parse_success",
                "original_correct", "number_is_given", "is_last_number_in_step",
                "control_answer_changed"):
        if col in frame:
            frame[col] = frame[col].map(_to_bool_or_nan)
    if "control_adjusted_change" in frame:
        frame["control_adjusted_change"] = pd.to_numeric(frame["control_adjusted_change"], errors="coerce")
    frame["position"] = frame["relative_position"]
    frame.loc[frame["perturbation"] == "truncation", "position"] = frame.loc[
        frame["perturbation"] == "truncation", "kept_fraction"]
    return frame


def bootstrap_mean(values, n_boot=2000, seed=0):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(x) == 0: return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return float(x.mean()), *np.percentile(means, [2.5, 97.5])


def summarize(frame, by, value="answer_changed", n_boot=2000):
    """Бутстрэп по вопросам; не фильтрует строки по format_ok."""
    by = list(by)
    keys = list(dict.fromkeys(by + ["question_id", "dataset", "model", "mode"]))
    per_question = frame.groupby(keys, dropna=False, observed=True)[value].mean().reset_index()
    rows = []
    for key, group in per_question.groupby(by, dropna=False, observed=True):
        key = key if isinstance(key, tuple) else (key,)
        mean, lo, hi = bootstrap_mean(group[value], n_boot=n_boot)
        rows.append({**dict(zip(by, key)), "n_questions": group[value].notna().sum(),
                     value: mean, "ci_lo": lo, "ci_hi": hi})
    return pd.DataFrame(rows)


def load_all_results(files):
    """Читает файлы всех моделей и объединяет; столбец model хранит имя модели, model_tag - тег."""
    frames = []
    for tag, path in files.items():
        if os.path.exists(path) and os.path.getsize(path) > 0:
            frame = load_results(path)
            frame["model_tag"] = tag
            frames.append(frame)
        else:
            print(f"Нет результатов для {tag}: {path}")
    if not frames:
        raise FileNotFoundError(
            "Нет ни одного файла результатов. Сначала выполните ячейку запуска run_all_models() "
            "и проверьте таблицу run_status (статус каждой модели).")
    return pd.concat(frames, ignore_index=True)


df = load_all_results(RESULT_FILES)
print("Строк:", len(df), "| вопросов:", df[["question_id", "dataset", "model", "mode"]].drop_duplicates().shape[0])
display(df.groupby("model_tag").size().rename("rows").to_frame())
print("Доля извлечённых ответов:", round(df["parse_success"].mean(), 3))
print("Доля строгого формата Answer:", round(df["format_ok"].mean(), 3))


Строк: 18979 | вопросов: 738


,rows
model_tag,
0.5B,3802
3B,7900
7B,7277


Доля извлечённых ответов: 0.983
Доля строгого формата Answer: 0.936


In [12]:
# Основные метрики. Не отбрасываем format_ok=False: fallback-ответы учитываются,
# а случаи без извлечённого ответа отдельно видны через parse_success.
summary = summarize(df, ["model", "dataset", "mode", "perturbation"], "answer_changed")
parse_summary = df.groupby(["model", "dataset", "mode", "perturbation"], dropna=False).agg(
    n_trials=("question_id", "size"), parse_success_rate=("parse_success", "mean"),
    strict_format_rate=("format_ok", "mean"),
    regenerated_accuracy=("regenerated_correct", "mean")
).reset_index()
summary.merge(parse_summary, on=["model", "dataset", "mode", "perturbation"], how="left")


,model,dataset,mode,perturbation,n_questions,answer_changed,ci_lo,ci_hi,n_trials,parse_success_rate,strict_format_rate,regenerated_accuracy
0,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,adjacent_swap,37,0.089146,0.037194,0.148649,235,0.914894,0.914894,0.911628
1,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,control,35,0.000000,0.000000,0.000000,38,0.921053,0.921053,1.000000
2,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,number_substitution,7,0.000000,0.000000,0.000000,55,0.981818,0.981818,1.000000
3,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,step_permutation,38,0.307018,0.197368,0.416667,114,0.903509,0.903509,0.679612
4,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,truncation,38,0.409777,0.335651,0.485937,273,0.967033,0.967033,0.587121
5,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,adjacent_swap,28,0.102381,0.028542,0.196488,124,1.000000,1.000000,0.895161
6,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,control,28,0.035714,0.000000,0.107143,28,1.000000,1.000000,0.964286
7,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,number_substitution,4,0.000000,0.000000,0.000000,16,1.000000,1.000000,1.000000
8,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,step_permutation,28,0.315476,0.184524,0.464435,80,0.987500,0.987500,0.670886
9,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,truncation,28,0.254238,0.157509,0.356434,152,0.993421,0.993421,0.721854


In [13]:
# Парное сравнение каждого вмешательства с неизменённым контролем того же вопроса.
# control_adjusted_change = changed(perturbed) - changed(control), диапазон [-1, 1].
paired = df[df["perturbation"] != "control"].copy()
paired_ci = summarize(paired, ["model", "dataset", "mode", "perturbation"], "control_adjusted_change")
paired_counts = paired.groupby(["model", "dataset", "mode", "perturbation"], dropna=False).agg(
    n_trials=("question_id", "size"),
    perturbed_change_rate=("answer_changed", "mean"),
    control_change_rate=("control_answer_changed", "mean"),
    parse_success_rate=("parse_success", "mean")
).reset_index()
paired_ci.merge(paired_counts, on=["model", "dataset", "mode", "perturbation"], how="left")



,model,dataset,mode,perturbation,n_questions,control_adjusted_change,ci_lo,ci_hi,n_trials,perturbed_change_rate,control_change_rate,parse_success_rate
0,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,adjacent_swap,35,0.094240,0.040539,0.155475,235,0.088372,0.000000,0.914894
1,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,number_substitution,7,0.000000,0.000000,0.000000,55,0.000000,0.000000,0.981818
2,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,step_permutation,35,0.333333,0.223810,0.442857,114,0.320388,0.000000,0.903509
3,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,truncation,35,0.409527,0.329526,0.492883,273,0.412879,0.000000,0.967033
4,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,adjacent_swap,28,0.066667,0.019048,0.130952,124,0.104839,0.040323,1.000000
5,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,number_substitution,4,0.000000,0.000000,0.000000,16,0.000000,0.000000,1.000000
6,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,step_permutation,28,0.279762,0.148810,0.434524,80,0.329114,0.037500,0.987500
7,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,truncation,28,0.218523,0.112229,0.330052,152,0.278146,0.039474,0.993421
8,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,cot,adjacent_swap,37,0.000000,0.000000,0.000000,213,0.000000,0.000000,1.000000
9,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,cot,number_substitution,37,0.061440,0.053715,0.068929,657,0.056317,0.000000,1.000000


In [14]:
# Позиционные эффекты: позиция шага и доля сохранённой цепочки анализируются раздельно.
numsub = df[df["perturbation"] == "number_substitution"]
summarize(numsub, ["model", "dataset", "mode", "relative_position", "number_is_given", "is_last_number_in_step"])


,model,dataset,mode,relative_position,number_is_given,is_last_number_in_step,n_questions,answer_changed,ci_lo,ci_hi
0,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,0.000000,1.0,0.0,1,0.000000,0.000000,0.00000
1,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,0.000000,1.0,1.0,1,0.000000,0.000000,0.00000
2,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,0.100000,1.0,1.0,1,0.000000,0.000000,0.00000
3,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,0.142857,0.0,0.0,1,0.000000,0.000000,0.00000
4,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,0.142857,0.0,1.0,1,0.000000,0.000000,0.00000
...,...,...,...,...,...,...,...,...,...,...
890,Qwen/Qwen2.5-7B-Instruct,gsm8k,fewshot,0.888889,0.0,1.0,1,0.000000,0.000000,0.00000
891,Qwen/Qwen2.5-7B-Instruct,gsm8k,fewshot,1.000000,0.0,0.0,75,0.000000,0.000000,0.00000
892,Qwen/Qwen2.5-7B-Instruct,gsm8k,fewshot,1.000000,0.0,1.0,81,0.839506,0.753086,0.91358
893,Qwen/Qwen2.5-7B-Instruct,gsm8k,fewshot,1.000000,1.0,0.0,47,0.000000,0.000000,0.00000


In [15]:
# Расширенные метрики: модель, размер модели, тип задачи и позиция вмешательства.
# Для размера модели желательно задать явный столбец model_size в исходном CSV
# (например: 0.5B, 3B, 7B). Автоматическая эвристика ниже — только fallback.
import re
import numpy as np
import pandas as pd

def infer_model_size(model_name):
    match = re.search(r"(\d+(?:\.\d+)?)\s*B\b", str(model_name), flags=re.I)
    return match.group(1) + "B" if match else "unknown"

def prepare_extended_metrics(frame):
    data = frame.copy()
    if "task_type" not in data.columns:
        # Датасет используется как приближение типа задачи, если отдельного поля нет.
        data["task_type"] = data["dataset"]
    if "model_size" not in data.columns:
        data["model_size"] = data["model"].map(infer_model_size)
    data["position"] = data["relative_position"].fillna(data["kept_fraction"])
    data["position_band"] = pd.cut(
        data["position"], bins=[-0.001, 1/3, 2/3, 1.001],
        labels=["начало", "середина", "конец"], include_lowest=True
    )
    return data

def metric_table(frame, group_cols):
    """Усреднение сначала по вопросу, затем по группам; единица бутстрэпа — вопрос."""
    cols = list(dict.fromkeys(group_cols + ["question_id", "model", "mode"]))
    per_question = frame.groupby(cols, dropna=False, observed=True).agg(
        answer_changed=("answer_changed", "mean"),
        regenerated_correct=("regenerated_correct", "mean")
    ).reset_index()
    return per_question.groupby(group_cols, dropna=False, observed=True).agg(
        n_questions=("question_id", "nunique"),
        answer_changed_rate=("answer_changed", "mean"),
        regenerated_accuracy=("regenerated_correct", "mean")
    ).reset_index()

extended = prepare_extended_metrics(df)

# 1. Общая устойчивость: тип вмешательства × тип задачи × размер модели.
metrics_by_model_task = metric_table(
    extended, ["model", "model_size", "task_type", "perturbation"]
)
display(metrics_by_model_task.sort_values(
    ["task_type", "model_size", "perturbation"]
))

# 2. Зависимость от позиции вмешательства.
metrics_by_position = metric_table(
    extended[extended["perturbation"].isin(
        ["number_substitution", "adjacent_swap", "truncation"]
    )],
    ["model_size", "task_type", "perturbation", "position_band"]
)
display(metrics_by_position.sort_values(
    ["task_type", "model_size", "perturbation", "position_band"]
))

# 3. Отдельная сводка: насколько вмешательство ухудшило точность относительно контроля.
control = metric_table(
    extended[extended["perturbation"] == "control"],
    ["model", "task_type", "perturbation"]
)
display(control)


,model,model_size,task_type,perturbation,n_questions,answer_changed_rate,regenerated_accuracy
0,Qwen/Qwen2.5-0.5B-Instruct,0.5B,arc,adjacent_swap,51,0.094847,0.905153
1,Qwen/Qwen2.5-0.5B-Instruct,0.5B,arc,control,51,0.015873,0.984127
2,Qwen/Qwen2.5-0.5B-Instruct,0.5B,arc,number_substitution,10,0.000000,1.000000
3,Qwen/Qwen2.5-0.5B-Instruct,0.5B,arc,step_permutation,51,0.310606,0.689394
4,Qwen/Qwen2.5-0.5B-Instruct,0.5B,arc,truncation,51,0.343791,0.656209
10,Qwen/Qwen2.5-3B-Instruct,3B,arc,adjacent_swap,86,0.006857,0.993143
11,Qwen/Qwen2.5-3B-Instruct,3B,arc,control,86,0.000000,1.000000
12,Qwen/Qwen2.5-3B-Instruct,3B,arc,number_substitution,23,0.011213,0.988787
13,Qwen/Qwen2.5-3B-Instruct,3B,arc,step_permutation,86,0.015015,0.984985
14,Qwen/Qwen2.5-3B-Instruct,3B,arc,truncation,86,0.108309,0.891691


,model_size,task_type,perturbation,position_band,n_questions,answer_changed_rate,regenerated_accuracy
0,0.5B,arc,adjacent_swap,начало,51,0.078205,0.921795
1,0.5B,arc,adjacent_swap,середина,50,0.064516,0.935484
2,0.5B,arc,adjacent_swap,конец,47,0.226415,0.773585
3,0.5B,arc,number_substitution,начало,9,0.000000,1.000000
4,0.5B,arc,number_substitution,середина,4,0.000000,1.000000
5,0.5B,arc,number_substitution,конец,3,0.000000,1.000000
6,0.5B,arc,truncation,начало,51,0.440404,0.559596
7,0.5B,arc,truncation,середина,51,0.328283,0.671717
8,0.5B,arc,truncation,конец,47,0.172619,0.827381
18,3B,arc,adjacent_swap,начало,86,0.004505,0.995495


,model,task_type,perturbation,n_questions,answer_changed_rate,regenerated_accuracy
0,Qwen/Qwen2.5-0.5B-Instruct,arc,control,51,0.015873,0.984127
1,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,control,56,0.000000,1.000000
2,Qwen/Qwen2.5-3B-Instruct,arc,control,86,0.000000,1.000000
3,Qwen/Qwen2.5-3B-Instruct,gsm8k,control,97,0.000000,1.000000
4,Qwen/Qwen2.5-7B-Instruct,arc,control,96,0.000000,1.000000
5,Qwen/Qwen2.5-7B-Instruct,gsm8k,control,95,0.000000,1.000000


## Интерпретация и ограничения

- Все варианты генерируются в одном режиме `answer`: модель получает полную изменённую цепочку и продолжает после `Answer:`.
- `answer_changed` сравнивает извлечённый ответ с исходным ответом. Если исходный или новый ответ отсутствует, значение остаётся пропущенным, а причина указана в `answer_change_status`.
- `parse_success` и `format_ok` показывают соответственно успешность извлечения и соблюдение строгого формата. Строки с `format_ok=False` не удаляются из основного анализа.
- `control_adjusted_change` — парная разность индикаторов изменения ответа между вмешательством и контролем того же вопроса.
- Для чисел из условия используются только отдельные поля `question_text`, `question`, `problem` или `query`. Если такого поля нет, `number_is_given` остаётся неизвестным; полный few-shot prompt не сканируется.
- Для обрезки `kept_fraction` — доля сохранённых первых шагов. Для замен и соседних перестановок `relative_position` — нормированная позиция шага.
- Эксперимент измеряет чувствительность ответа к переданному тексту цепочки, но не доказывает наличие или использование внутреннего CoT.


## Проверка корректности модуля подмены

1. **Инварианты на уровне строк**: каждая подмена должна менять цепочку ровно так, как заявлено.
2. **format_ok**: сколько вариантов выпадает из таблиц из-за формата ответа и не смещает ли это оценки.
3. **control**: воспроизводит ли продолжение немодифицированного префикса исходный хвост цепочки.
4. **Эвристика number_is_given**: не захватывает ли она числа из few-shot примеров.

In [16]:
# Проверка инвариантов каждого вида вмешательства.
raw = pd.concat(
    [pd.read_csv(path, encoding="utf-8-sig",
                 dtype={"old_number": str, "new_number": str, "question_id": str}).assign(model_tag=tag)
     for tag, path in RESULT_FILES.items() if os.path.exists(path) and os.path.getsize(path) > 0],
    ignore_index=True)
raw["orig"] = raw["original_steps"].map(json.loads)
raw["pert"] = raw["perturbed_steps"].map(json.loads)


def _num(value):
    return float(str(value).replace(",", ""))


def check_row(r):
    o, p, kind = r["orig"], r["pert"], r["perturbation"]
    n = len(o)
    if kind == "control":
        return p == o
    if kind == "truncation":
        keep = int(r["truncation_boundary"])
        return p == o[:keep] and len(p) == keep and keep < n
    if kind == "adjacent_swap":
        i = int(r["target_step_index"])
        if not 0 <= i < n-1: return False
        expected = list(o); expected[i], expected[i+1] = expected[i+1], expected[i]
        return p == expected and p != o
    if kind == "step_permutation":
        try: order = json.loads(r["permutation_order"])
        except Exception: return False
        return (sorted(order) == list(range(n)) and p == [o[i] for i in order]
                and p != o and sorted(p) == sorted(o))
    if kind == "number_substitution":
        i, k = int(r["target_step_index"]), int(r["target_number_index"])
        if len(p) != n or any(p[j] != o[j] for j in range(n) if j != i): return False
        matches = find_numbers(o[i])
        if not 0 <= k < len(matches): return False
        m = matches[k]; old, new = str(r["old_number"]), str(r["new_number"])
        same_sign = _num(old) == 0 or _num(old) * _num(new) > 0
        expected = o[i][:m.start()] + new + o[i][m.end():]
        return m.group() == old and new != old and same_sign and p[i] == expected
    return False

raw["invariant_ok"] = raw.apply(check_row, axis=1)
display(raw.groupby(["model", "perturbation"])["invariant_ok"].agg(rows="size", share_ok="mean"))



rows  share_ok
model                      perturbation                       
Qwen/Qwen2.5-0.5B-Instruct adjacent_swap         803       1.0
                           control               148       1.0
                           number_substitution  1460       1.0
                           step_permutation      440       1.0
                           truncation            951       1.0
Qwen/Qwen2.5-3B-Instruct   adjacent_swap        1455       1.0
                           control               290       1.0
                           number_substitution  3568       1.0
                           step_permutation      842       1.0
                           truncation           1745       1.0
Qwen/Qwen2.5-7B-Instruct   adjacent_swap        1183       1.0
                           control               300       1.0
                           number_substitution  3463       1.0
                           step_permutation      848       1.0
                           truncation           1483       1.0

In [17]:
# Доля успешного извлечения и строгого формата по видам вмешательства.
fmt = df.groupby(["model", "dataset", "mode", "perturbation"], dropna=False).agg(
    rows=("question_id", "size"), parse_success_rate=("parse_success", "mean"),
    strict_format_rate=("format_ok", "mean"),
    unparseable=("answer_change_status", lambda s: (s == "unparseable_regenerated").sum())
).reset_index()
display(fmt)


,model,dataset,mode,perturbation,rows,parse_success_rate,strict_format_rate,unparseable
0,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,adjacent_swap,235,0.914894,0.914894,20
1,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,control,38,0.921053,0.921053,3
2,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,number_substitution,55,0.981818,0.981818,1
3,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,step_permutation,114,0.903509,0.903509,11
4,Qwen/Qwen2.5-0.5B-Instruct,arc,cot,truncation,273,0.967033,0.967033,9
5,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,adjacent_swap,124,1.000000,1.000000,0
6,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,control,28,1.000000,1.000000,0
7,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,number_substitution,16,1.000000,1.000000,0
8,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,step_permutation,80,0.987500,0.987500,1
9,Qwen/Qwen2.5-0.5B-Instruct,arc,fewshot,truncation,152,0.993421,0.993421,1


In [18]:
# Проверка парности: на каждый вопрос должен приходиться ровно один контроль.
control_counts = raw[raw["perturbation"] == "control"].groupby(
    ["question_id", "dataset", "model", "mode"]
).size()
print("Контролей на вопрос (распределение):")
display(control_counts.value_counts().rename_axis("n_controls").to_frame("n_questions"))


Контролей на вопрос (распределение):


,n_questions
n_controls,
1,738


In [19]:
# Доступность классификации чисел исходного условия.
ns = df[df["perturbation"] == "number_substitution"]
display(ns.groupby(["model", "dataset", "mode"], dropna=False)["number_is_given"]
          .agg(rows="size", known="count", share_given="mean").round(3))
print("Если known=0, добавьте в исходный CSV отдельный столбец question_text.")


rows  known  share_given
model                      dataset mode                             
Qwen/Qwen2.5-0.5B-Instruct arc     cot        55     55        0.618
                                   fewshot    16     16        0.625
                           gsm8k   cot       657    657        0.505
                                   fewshot   732    732        0.471
Qwen/Qwen2.5-3B-Instruct   arc     cot       132    132        0.538
                                   fewshot    71     71        0.845
                           gsm8k   cot      1909   1909        0.484
                                   fewshot  1456   1456        0.496
Qwen/Qwen2.5-7B-Instruct   arc     cot        82     82        0.598
                                   fewshot    34     34        0.882
                           gsm8k   cot      1845   1845        0.493
                                   fewshot  1502   1502        0.491

Если known=0, добавьте в исходный CSV отдельный столбец question_text.


In [20]:
# Сравнение моделей: прирост доли изменённых ответов относительно контроля (control_adjusted_change)
compare = summarize(df[df["perturbation"] != "control"], ["dataset", "perturbation", "model"], "control_adjusted_change")
display(compare.pivot_table(index=["dataset", "perturbation"], columns="model",
                            values="control_adjusted_change").round(3))
display(compare.pivot_table(index=["dataset", "perturbation"], columns="model",
                            values="n_questions"))

model                        Qwen/Qwen2.5-0.5B-Instruct  \
dataset perturbation                                      
arc     adjacent_swap                             0.082   
        number_substitution                       0.000   
        step_permutation                          0.310   
        truncation                                0.325   
gsm8k   adjacent_swap                             0.000   
        number_substitution                       0.062   
        step_permutation                          0.122   
        truncation                                0.724   

model                        Qwen/Qwen2.5-3B-Instruct  \
dataset perturbation                                    
arc     adjacent_swap                           0.007   
        number_substitution                     0.011   
        step_permutation                        0.015   
        truncation                              0.108   
gsm8k   adjacent_swap                           0.002   
        number_substitution                     0.038   
        step_permutation                        0.013   
        truncation                              0.656   

model                        Qwen/Qwen2.5-7B-Instruct  
dataset perturbation                                   
arc     adjacent_swap                           0.002  
        number_substitution                     0.010  
        step_permutation                        0.000  
        truncation                              0.164  
gsm8k   adjacent_swap                           0.005  
        number_substitution                     0.047  
        step_permutation                        0.015  
        truncation                              0.816

model                        Qwen/Qwen2.5-0.5B-Instruct  \
dataset perturbation                                      
arc     adjacent_swap                              63.0   
        number_substitution                        11.0   
        step_permutation                           63.0   
        truncation                                 63.0   
gsm8k   adjacent_swap                              82.0   
        number_substitution                        82.0   
        step_permutation                           82.0   
        truncation                                 82.0   

model                        Qwen/Qwen2.5-3B-Instruct  \
dataset perturbation                                    
arc     adjacent_swap                           111.0   
        number_substitution                      28.0   
        step_permutation                        111.0   
        truncation                              111.0   
gsm8k   adjacent_swap                           179.0   
        number_substitution                     179.0   
        step_permutation                        179.0   
        truncation                              179.0   

model                        Qwen/Qwen2.5-7B-Instruct  
dataset perturbation                                   
arc     adjacent_swap                           117.0  
        number_substitution                      21.0  
        step_permutation                        117.0  
        truncation                              116.0  
gsm8k   adjacent_swap                           183.0  
        number_substitution                     183.0  
        step_permutation                        183.0  
        truncation                              183.0